## Download dependencies

In [2]:
!pip install vizdoom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.9/38.9 MB 40.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 43.2 MB/s eta 0:00:0000:0100:01


## Import libraries

In [3]:
import gymnasium as gym 
import torch 
import torch.nn as nn
import torch.nn.functional
from torch.optim import AdamW 
import numpy as np 
from vizdoom import gymnasium_wrapper
from dataclasses import dataclass

print("Success")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# we will have a global h_t and c_t tensor so they are stateless
hidden_state_tensor, memory_tensor = None, None


Success


## Initialisation

In [4]:
class InputError(Exception):
    """
    Custom exception raised when input is invalid.
    """
    def __init__(self, fan_in, message: str ="Error! Input tensor does not match expected shape : "):
        self.size = fan_in
        self.message = message
        super().__init__(f"{self.message} {self.size}")


class CNN(nn.Module):
    """
    Simple CNN. Takes an image of size 128x128 ( shape is (3, 128, 128) )
    and turns it into -> (1024, 4, 4) 
    """
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels = 3, out_channels = 64, kernel_size = 4, stride = 2, padding = 1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size = 4, stride = 2, padding = 1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.Conv2d(in_channels = 128, out_channels = 256, kernel_size = 4, stride = 2, padding = 1),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.Conv2d(in_channels = 256, out_channels = 512, kernel_size = 4, stride = 2, padding = 1),
            # nn.BatchNorm2d(512),
            # nn.GELU(),
            # nn.Conv2d(in_channels = 512, out_channels = 1024, kernel_size = 4, stride = 2, padding = 1)
        )
        #self.opt = AdamW(self.parameters(), weight_decay = 1e-2, lr = 1e-3)
        #self.loss = None

    def forward(self, x):
        return self.layers(x)


class LSTMCELL(nn.Module):
    """
    Long Short-Term memory cell. Used by LSTM since it is its foundational block.
    Takes as input zt = [ xt ] and computes : forget_gate = how much to forget from old memory?
                        [ht-1]                input_gate = how much new info to store?
                                              memory_available = how much new info can we store?
                                              output = what we actually learn and remember
    ht = hidden state at moment=t, ct= memory of what happened at moment=t
    """
    def __init__(self, fan_in: int, hidden: int = 128):
        super().__init__()
        # # self.input = input_tensor
        # self.h, self.c = torch.randn(hidden), torch.randn(hidden)
        self.fan_in, self.hidden = fan_in, hidden
        
        self.forget = nn.Linear(fan_in + hidden, hidden)
        self.input = nn.Linear(fan_in + hidden, hidden)
        self.memory_available = nn.Linear(fan_in + hidden, hidden)
        self.output = nn.Linear(fan_in + hidden, hidden)
        
        self.layers = [self.forget, self.input, self.memory_available, self.output]
        
    def forward(self, input_tensor, h, c):
        if input_tensor.shape[-1] != self.fan_in:
            raise InputError(self.fan_in)

        # creates z_t using h_t-1 and x_t but this is not fully correct yet
        # each cell must have its own ht and ct
        final_input = torch.cat((input_tensor, h), dim=-1)
            
        c = torch.sigmoid(self.forget(final_input)) * c + \
        torch.sigmoid(self.input(final_input)) * torch.tanh(self.memory_available(final_input))
        
        h = torch.sigmoid(self.output(final_input)) * torch.tanh(c)

        return (h, c)

class LSTM(nn.Module):
    """
    LSTM RNN. Takes image, flattens it, then passes it through each LSTM cell.
    """
    def __init__(self, input_x, hidden_size: int = 128, cell_count: int = 1,
                 final_number: int = 10):
        super().__init__()
        # a preprocess phase, dynamically changing the input to smaller size
        self.preprocess_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_x.shape[-1], hidden_size),
            nn.GELU()
        )
        self.cells = nn.ModuleList(
            [LSTMCELL(fan_in = hidden_size) for _ in range(cell_count)]
        )
        self.cc = cell_count
        self.h = hidden_size
        # just to convert into logits
        self.policy_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, final_number)
        )

    def forward(self, x, h, c):
        x = self.preprocess_layers(x)

        for idx in range(self.cc):

            h, c = self.cells[idx](x, h, c)
            x = h

        return self.policy_head(h), h, c
        

class InputWrapper(gym.ObservationWrapper):
    """
    Custom environment wrapper, this just converts the imgs to 128x128
    """
    def __init__(self, env: gym.Env, img_size: int = 64):
        super().__init__(env)
        self.img_size = img_size
        self.observation_space = env.observation_space
        self.observation_space["screen"] = gym.spaces.Box(
            low = self.screen_observation(self.observation_space["screen"].low),
            high = self.screen_observation(self.observation_space["screen"].high),
            dtype = np.float32
        )

    def screen_observation(self, observation):
        from cv2 import resize
        # if isinstance(observation, dict):
        #     observation = observation["screen"]

        observation = resize(observation, (self.img_size, self.img_size))
        observation = np.moveaxis(observation, -1, 0)

        return observation.astype(np.float32)

    def observation(self, observation):
        observation["screen"] = self.screen_observation(observation["screen"])
        return observation

@dataclass
class EpisodeStep:
    observation: np.ndarray
    action: int

@dataclass
class Episode:
    steps: list[EpisodeStep]
    reward: float

print("Success")

Success


## Functionalities

In [13]:
def generate_data(env: gym.Env, cnn, lstm, ep_count: int = 70):
    """
    Generate infinite data, take ep_count episodes and use them after to filter for elites
    """
    global device
    global hidden_state_tensor, memory_tensor
    
    total_episodes, episode_steps = [], []
    total_reward = 0
    #env = InputWrapper(env)
    obs, _ = env.reset()
    flat, softmax = nn.Flatten(), nn.Softmax(dim = -1)
    hidden_state_tensor, memory_tensor = torch.zeros(1, lstm.h, device=device), torch.zeros(1, lstm.h, device=device)
    while True:
        with torch.no_grad():
            # sample an episode for its total reward
            screen_obs = torch.tensor(obs["screen"], dtype = torch.float32, device=device)
            screen_obs = screen_obs * 2 / 255.0 - 1.0
            #print(screen_obs.shape)
            cnn_out = cnn(screen_obs.unsqueeze(0))
            flat_out = flat(cnn_out)
            lstm_in = torch.cat( (torch.tensor(obs["gamevariables"], 
                                               dtype = torch.float32, device=device).unsqueeze(0),
                                 flat_out), dim = -1)
            #print(lstm_in.shape)
            
            lstm_out, hidden_state_tensor, memory_tensor = lstm(lstm_in, hidden_state_tensor,
                                                               memory_tensor)
            action_probs = softmax(lstm_out / 2)
            action = torch.multinomial(action_probs, 1).item()

        new_obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        episode_steps.append(EpisodeStep(observation = obs, action = action))
        
        if terminated or truncated:
            # if its over we keep track
            if total_reward >= -50:
                total_episodes.append( Episode(reward = total_reward, 
                                              steps = episode_steps.copy() ) )
            hidden_state_tensor.zero_()
            memory_tensor.zero_()
            episode_steps.clear()
            obs, _ = env.reset()
            total_reward = 0
            
            if len(total_episodes) >= ep_count:
                yield total_episodes
                total_episodes.clear()            
        else:
            obs = new_obs

def filter_episodes(episodes: list[Episode], reward_threshold: float):
    # to have memory over entire episode instead of per frame we dont squish all episodes together
    rewards = [e.reward for e in episodes]

    bound = np.percentile(rewards, reward_threshold * 100)
    mean = np.mean(rewards)

    elite_episodes = []

    for episode in episodes:
        if episode.reward >= bound:
            elite_episodes.append(episode)

    return elite_episodes, mean

print("Success")

Success


# Main

In [14]:
env = gym.make("VizdoomBasic-v1")
env = InputWrapper(env)
possible_actions = env.action_space.n
cnn_model = CNN().to(device)
sample_init, _ = env.reset()
sample = cnn_model(torch.tensor(sample_init["screen"], device=device).unsqueeze(0))
flat = nn.Flatten()
sample = flat(sample)
print(sample.shape)

sample = torch.cat( (torch.tensor(sample_init["gamevariables"], dtype = torch.float32,
                                  device=device).unsqueeze(0), sample), dim = -1)

lstm_model = LSTM(input_x = sample, final_number = possible_actions).to(device)

loss = nn.CrossEntropyLoss()
best_reward = -float('inf')

optimizer = AdamW(list(cnn_model.parameters()) + list(lstm_model.parameters()), weight_decay = 0.02, lr=1e-3)
for iter_no, batch in enumerate(generate_data(env, cnn_model, lstm_model)):

    episodes, reward_mean = filter_episodes(batch, 0.7)

    if abs(reward_mean - best_reward) <= 1e-4:
        break

    if reward_mean > best_reward:
        best_reward = reward_mean

    episode_loss = 0.0
    valueloss = None
    for episode in episodes:

        optimizer.zero_grad()
        episode_loss = 0.0
        
        hidden_state_tensor = torch.zeros(1, lstm_model.h, device=device)
        memory_tensor = torch.zeros(1, lstm_model.h, device=device)
        count = 0
        
        for i, step in enumerate(episode.steps):
            count += 1
            screen = torch.tensor(step.observation["screen"],
                                  dtype=torch.float32, device=device).unsqueeze(0)
            screen = screen * 2 / 255.0 - 1.0
            
            game_vars = torch.tensor(step.observation["gamevariables"], 
                                     dtype=torch.float32, device=device).unsqueeze(0)

            action = torch.tensor([step.action], dtype=torch.long, device=device)

            img_v = cnn_model(screen)
            img_v = flat(img_v)

            lstm_in = torch.cat((game_vars, img_v), dim=-1)

            logits, hidden_state_tensor, memory_tensor = lstm_model(lstm_in, 
                                                                    hidden_state_tensor, 
                                                                    memory_tensor)

            episode_loss += loss(logits, action)
            
            if (i + 1) % 16 == 0 or i == len(episode.steps) - 1:
                episode_loss /= count
                valueloss = episode_loss.item()
            
                episode_loss.backward()

                torch.nn.utils.clip_grad_norm_(list(cnn_model.parameters()) + list(lstm_model.parameters()),
                                               max_norm=1.0)
                
                optimizer.step()
                optimizer.zero_grad()
        
                hidden_state_tensor = hidden_state_tensor.detach()
                memory_tensor = memory_tensor.detach()
        
                episode_loss = 0.0
                count = 0
        
        if iter_no % 10 == 0:
            print(f"Iteration {iter_no} | Loss {valueloss:.4f} | Mean reward {reward_mean:.5f}")


torch.Size([1, 8192])
Iteration 0 | Loss 1.4061 | Mean reward 57.20000
Iteration 0 | Loss 1.4271 | Mean reward 57.20000
Iteration 0 | Loss 1.4473 | Mean reward 57.20000
Iteration 0 | Loss 1.4121 | Mean reward 57.20000
Iteration 0 | Loss 1.3969 | Mean reward 57.20000
Iteration 0 | Loss 1.3669 | Mean reward 57.20000
Iteration 0 | Loss 1.3851 | Mean reward 57.20000
Iteration 0 | Loss 1.3850 | Mean reward 57.20000
Iteration 0 | Loss 1.3301 | Mean reward 57.20000
Iteration 0 | Loss 1.3408 | Mean reward 57.20000
Iteration 0 | Loss 1.3194 | Mean reward 57.20000
Iteration 0 | Loss 1.3320 | Mean reward 57.20000
Iteration 0 | Loss 1.3603 | Mean reward 57.20000
Iteration 0 | Loss 1.3087 | Mean reward 57.20000
Iteration 0 | Loss 1.3519 | Mean reward 57.20000
Iteration 0 | Loss 1.6493 | Mean reward 57.20000
Iteration 0 | Loss 1.4211 | Mean reward 57.20000
Iteration 0 | Loss 1.5917 | Mean reward 57.20000
Iteration 0 | Loss 1.3183 | Mean reward 57.20000
Iteration 0 | Loss 1.3836 | Mean reward 57.2000

KeyboardInterrupt: 

# Testing

In [15]:
import imageio
from IPython.display import Video

cnn_model.eval()
lstm_model.eval()

env = gym.make("VizdoomBasic-v1")
env = InputWrapper(env)

obs, _ = env.reset()

frames = []
done = False

flat = nn.Flatten()

while not done:
    frames.append(np.moveaxis(obs["screen"], 0, -1).astype(np.uint8))

    with torch.no_grad():
        screen = torch.tensor(
            obs["screen"],
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)

        cnn_out = cnn_model(screen)
        cnn_out = flat(cnn_out)

        lstm_in = torch.cat(
            (
                torch.tensor(
                    obs["gamevariables"],
                    dtype=torch.float32,
                    device=device
                ).unsqueeze(0),
                cnn_out
            ),
            dim=-1
        )

        logits, hidden_state_tensor, memory_tensor = lstm_model(lstm_in, 
                                                                    hidden_state_tensor, 
                                                                    memory_tensor)

        action = torch.argmax(logits, dim=-1).item()

    obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

env.close()

imageio.mimsave("/kaggle/working/doom.mp4", frames, fps=35)

Video("/kaggle/working/doom.mp4", embed=True)